# Exchange rates: rolling averages

This notebook reads ECB exchange-rate data staged in HDFS, computes rolling
averages per currency, and writes the aggregated table back to HDFS. It runs
on Spark in client mode (the notebook is the driver) against the Kerberized
HDFS cluster, authenticating with the `spark` principal in the `KNAB.COM` realm.

We launch Spark on Kubernetes with the notebook acting as the driver. The
executor image must match the driver's Spark/Python versions. Kerberos
credentials are mounted at `/stackable/kerberos`, so the session can
authenticate to HDFS.

In [1]:
import os
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

POD_NAME = os.environ["HOSTNAME"]
spark = (
    SparkSession.builder
    .master(f"k8s://https://{os.environ['KUBERNETES_SERVICE_HOST']}:{os.environ['KUBERNETES_SERVICE_PORT']}")
    .appName(f"lakehouse-{POD_NAME}")
    .config("spark.kubernetes.container.image", "oci.stackable.tech/demos/spark:3.5.2-python311")
    .config("spark.kubernetes.namespace", "default")
    .config("spark.kubernetes.authenticate.driver.serviceAccountName", "spark")
    .config("spark.kubernetes.authenticate.executor.serviceAccountName", "spark")
    .config("spark.kubernetes.driver.pod.name", POD_NAME)
    .config("spark.driver.port", "2222")
    .config("spark.driver.blockManager.port", "7777")
    .config("spark.executor.instances", "2")
    .config("spark.kerberos.principal", "spark/spark.default.svc.cluster.local@KNAB.COM")
    .config("spark.kerberos.keytab", "/stackable/kerberos/keytab")
    .config("spark.driver.extraJavaOptions", "-Djava.security.krb5.conf=/stackable/kerberos/krb5.conf")
    .config("spark.kubernetes.kerberos.krb5.path", "/stackable/kerberos/krb5.conf")
    .config("spark.executorEnv.KERBEROS_REALM", "KNAB.COM")
    .getOrCreate()
)

## Inspect the staged data

The default filesystem is `hdfs://hdfs/`. Before reading, we list the staging
directory to confirm the raw exchange-rate data is present.

In [2]:
hadoop = spark._jvm.org.apache.hadoop
fs = hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
for s in fs.listStatus(hadoop.fs.Path("/staging/exchange_rates/rates")):
    print(s.getPath())

hdfs://hdfs/staging/exchange_rates/rates/data.parquet


## Compute rolling averages

We filter to the annual (`P1Y`), end-of-period (`E`) rates denominated in EUR,
then compute trailing 1-, 3-, 5- and 10-year averages per currency. Finally we
express each year's rate as a percentage change against its 3-, 5- and 10-year
average.

The 1-year average equals the observed value by construction.

In [3]:
df = spark.read.parquet("hdfs://hdfs/staging/exchange_rates/rates")
base_rates = (
    df
    .filter(
        (F.col("time_format") == "P1Y") & 
        (F.col("collection") == "E") & 
        (F.col("currency_denom") == "EUR")
    )
    .select(
        F.col("time_period").cast("int").alias("time_period"),
        F.col("currency"),
        F.col("obs_value")
    )
)

window_spec = Window.partitionBy("currency").orderBy("time_period")

rolling_avg_1y = F.avg("obs_value").over(window_spec.rowsBetween(0, 0))
rolling_avg_3y = F.avg("obs_value").over(window_spec.rowsBetween(-2, 0))
rolling_avg_5y = F.avg("obs_value").over(window_spec.rowsBetween(-4, 0))
rolling_avg_10y = F.avg("obs_value").over(window_spec.rowsBetween(-9, 0))

rolling = base_rates.select(
    "time_period",
    "currency",
    "obs_value",
    rolling_avg_1y.alias("rolling_avg_1y"),
    rolling_avg_3y.alias("rolling_avg_3y"),
    rolling_avg_5y.alias("rolling_avg_5y"),
    rolling_avg_10y.alias("rolling_avg_10y")
)

final_df = rolling.withColumns({
    "pct_change_vs_3y_avg": ((F.col("obs_value") - F.col("rolling_avg_3y")) / F.col("rolling_avg_3y")) * 100,
    "pct_change_vs_5y_avg": ((F.col("obs_value") - F.col("rolling_avg_5y")) / F.col("rolling_avg_5y")) * 100,
    "pct_change_vs_10y_avg": ((F.col("obs_value") - F.col("rolling_avg_10y")) / F.col("rolling_avg_10y")) * 100
})

final_df.show()

+-----------+--------+---------+--------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|time_period|currency|obs_value|rolling_avg_1y|    rolling_avg_3y|    rolling_avg_5y|   rolling_avg_10y|pct_change_vs_3y_avg|pct_change_vs_5y_avg|pct_change_vs_10y_avg|
+-----------+--------+---------+--------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|       2000|     ARS|  0.92901|       0.92901|           0.92901|           0.92901|           0.92901|                 0.0|                 0.0|                  0.0|
|       2001|     ARS|  0.88001|       0.88001|0.9045099999999999|0.9045099999999999|0.9045099999999999|  -2.708648881714958|  -2.708648881714958|   -2.708648881714958|
|       2002|     ARS|  3.53936|       3.53936|1.7827933333333332|1.7827933333333332|1.7827933333333332|   98.52890033991602|   98.52890033991602|    98.52

## Write the aggregated table back to HDFS

The result is written to the aggregations staging area, overwriting any
previous run.

In [4]:
final_df.write.mode("overwrite").parquet("hdfs://hdfs/staging/aggregations/rates_to_euros_tb")

## Verify the written output

In [5]:
for s in fs.listStatus(hadoop.fs.Path("/staging/aggregations/rates_to_euros_tb")):
    print(s.getPath())

hdfs://hdfs/staging/aggregations/rates_to_euros_tb/_SUCCESS
hdfs://hdfs/staging/aggregations/rates_to_euros_tb/part-00000-7268a300-2d9f-44df-af56-f9053b29399b-c000.snappy.parquet


In [6]:
spark.read.parquet("hdfs://hdfs/staging/aggregations/rates_to_euros_tb").show()

+-----------+--------+---------+--------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|time_period|currency|obs_value|rolling_avg_1y|    rolling_avg_3y|    rolling_avg_5y|   rolling_avg_10y|pct_change_vs_3y_avg|pct_change_vs_5y_avg|pct_change_vs_10y_avg|
+-----------+--------+---------+--------------+------------------+------------------+------------------+--------------------+--------------------+---------------------+
|       2000|     ARS|  0.92901|       0.92901|           0.92901|           0.92901|           0.92901|                 0.0|                 0.0|                  0.0|
|       2001|     ARS|  0.88001|       0.88001|0.9045099999999999|0.9045099999999999|0.9045099999999999|  -2.708648881714958|  -2.708648881714958|   -2.708648881714958|
|       2002|     ARS|  3.53936|       3.53936|1.7827933333333332|1.7827933333333332|1.7827933333333332|   98.52890033991602|   98.52890033991602|    98.52